In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [39]:
df_pf = pd.read_csv('../data/portfolio.csv')
df_pro = pd.read_csv('../data/profile_ag.csv')
df_tran = pd.read_csv('../data/transcript.csv')

---

# profile.csv

## 1. 결측치 확인

- `gender`, `income`, `age` 컬럼의 결측치 수가 2175로 동일하여,<br>해당 결측치가 동일한 행에서 발생한 것인지 확인 

In [40]:
# 1. 조건 정의
cond_gender = df_pro['gender'].isnull()
cond_income = df_pro['income'].isnull()
cond_age = df_pro['age'] == 118

# 2. id 집합 생성
set_gender = set(df_pro.loc[cond_gender, 'id'])
set_income = set(df_pro.loc[cond_income, 'id'])
set_age = set(df_pro.loc[cond_age, 'id'])

# 3. 각각 개수 확인
print("gender 결측 수:", len(set_gender))
print("income 결측 수:", len(set_income))
print("age == 118 수:", len(set_age))
print("-" * 40)

# 4. 집합 비교 (완전 동일 여부)
print("gender == income:", set_gender == set_income)
print("gender == age:", set_gender == set_age)
print("income == age:", set_income == set_age)
print("-" * 40)

# 5. 교집합 확인
print("gender ∩ income:", len(set_gender & set_income))
print("gender ∩ age:", len(set_gender & set_age))
print("income ∩ age:", len(set_income & set_age))
print("-" * 40)

# 6. 차이 확인 (어디가 다른지)
print("gender - income:", len(set_gender - set_income))
print("income - gender:", len(set_income - set_gender))
print("gender - age:", len(set_gender - set_age))
print("age - gender:", len(set_age - set_gender))
print("-" * 40)

# 7. 완전 동일 패턴 여부 (3개 다 동일)
all_equal = (set_gender == set_income == set_age)
print("세 조건 모두 동일한 id 집합인지:", all_equal)

gender 결측 수: 2175
income 결측 수: 2175
age == 118 수: 2175
----------------------------------------
gender == income: True
gender == age: True
income == age: True
----------------------------------------
gender ∩ income: 2175
gender ∩ age: 2175
income ∩ age: 2175
----------------------------------------
gender - income: 0
income - gender: 0
gender - age: 0
age - gender: 0
----------------------------------------
세 조건 모두 동일한 id 집합인지: True


> 같은 행에서 발생함을 확인

## 2. gender 컬럼의 other

- `gender` 컬럼에서 `other` 값을 어떻게 처리할까

In [41]:
df_pro['gender'].value_counts(dropna=False, normalize=True).round(3)

gender
M      0.499
F      0.361
NaN    0.128
O      0.012
Name: proportion, dtype: float64

> `other`의 비율은 약 1%로 버려도 괜찮을 것 같음
<br> NaN 값을 어떻게 처리할지가 더 문제ㅠ

---

# transcript.csv

In [42]:
df_tran1 = df_tran.sort_values(['person', 'time'])
df_tran1.head(30)

,Unnamed: 0,person,event,value,time
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,{'offer id': '5a8bc65990b245e5a138643cd4eb9837'},168
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,{'offer id': '5a8bc65990b245e5a138643cd4eb9837'},192
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,{'amount': 22.16},228
113605,113605,0009655768c64bdeb2e877511632db8f,offer received,{'offer id': '3f207df678b143eea3cee63160fa8bed'},336
139992,139992,0009655768c64bdeb2e877511632db8f,offer viewed,{'offer id': '3f207df678b143eea3cee63160fa8bed'},372
153401,153401,0009655768c64bdeb2e877511632db8f,offer received,{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},408
168412,168412,0009655768c64bdeb2e877511632db8f,transaction,{'amount': 8.57},414
168413,168413,0009655768c64bdeb2e877511632db8f,offer completed,{'offer_id': 'f19421c1d4aa40978ebb69ca19b0e20d...,414
187554,187554,0009655768c64bdeb2e877511632db8f,offer viewed,{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},456
204340,204340,0009655768c64bdeb2e877511632db8f,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},504


In [43]:
df_tran[(df_tran['time'] == 0) & (df_tran['event'] == 'offer completed')]

,Unnamed: 0,person,event,value,time
12658,12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,offer completed,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...,0
12672,12672,fe97aa22dd3e48c8b143116a8403dd52,offer completed,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...,0
12679,12679,629fc02d56414d91bca360decdfa9288,offer completed,{'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...,0
12692,12692,676506bad68e4161b9bbaffeb039626b,offer completed,{'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...,0
12697,12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,offer completed,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...,0
...,...,...,...,...,...
15521,15521,2c107d718e614961ae18c8ef2af37b03,offer completed,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...,0
15532,15532,0b64be3b241c4407a5c9a71781173829,offer completed,{'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...,0
15536,15536,1593d617fac246ef8e50dbb0ffd77f5f,offer completed,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...,0
15541,15541,f367a50b86d049799bbb0eb645ee834c,offer completed,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...,0


---

In [44]:
# value 문자열 -> dict 변환
df_tran['value'] = df_tran['value'].apply(ast.literal_eval)

# offer_id 추출 ('offer id' 또는 'offer_id' 둘 다 대응)
df_tran['offer_id'] = df_tran['value'].apply(
    lambda x: x.get('offer id') if isinstance(x, dict) and 'offer id' in x
    else (x.get('offer_id') if isinstance(x, dict) and 'offer_id' in x
    else (x.get('amount') if isinstance(x, dict) and 'amount' in x else None))
)

df_tran = df_tran.drop(columns='value')
# df_tran['offer_id'] = df_tran['value'].apply(
#     lambda x: x.get('offer id') if isinstance(x, dict) else None
# )

In [45]:
df_tran1 = df_tran.sort_values(['person', 'time'])
df_tran1.head(3)

,Unnamed: 0,person,event,time,offer_id
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,5a8bc65990b245e5a138643cd4eb9837
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,5a8bc65990b245e5a138643cd4eb9837
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16


In [46]:
df_merge = df_tran.merge(
    df_pf[['offer_id', 'offer_type', 'reward', 'difficulty', 'duration']],
    on='offer_id',
    how='left'
)

df_merge.sort_values(['person', 'time']).head(1)

,Unnamed: 0,person,event,time,offer_id,offer_type,reward,difficulty,duration
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0.0,3.0


In [47]:
df_tran.loc[47582, 'person']

'78afa995795e4d85b5d9ceeca43f5fef'

In [48]:
person_id = '78afa995795e4d85b5d9ceeca43f5fef'

display(
    df_merge[df_merge['person'] == person_id]
    .sort_values('time')
)

,Unnamed: 0,person,event,time,offer_id,offer_type,reward,difficulty,duration
0,0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5.0,5.0,7.0
15561,15561,78afa995795e4d85b5d9ceeca43f5fef,offer viewed,6,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5.0,5.0,7.0
47582,47582,78afa995795e4d85b5d9ceeca43f5fef,transaction,132,19.89,NaN,NaN,NaN,NaN
47583,47583,78afa995795e4d85b5d9ceeca43f5fef,offer completed,132,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5.0,5.0,7.0
49502,49502,78afa995795e4d85b5d9ceeca43f5fef,transaction,144,17.78,NaN,NaN,NaN,NaN
53176,53176,78afa995795e4d85b5d9ceeca43f5fef,offer received,168,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0.0,3.0
85291,85291,78afa995795e4d85b5d9ceeca43f5fef,offer viewed,216,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0.0,3.0
87134,87134,78afa995795e4d85b5d9ceeca43f5fef,transaction,222,19.67,NaN,NaN,NaN,NaN
92104,92104,78afa995795e4d85b5d9ceeca43f5fef,transaction,240,29.72,NaN,NaN,NaN,NaN
141566,141566,78afa995795e4d85b5d9ceeca43f5fef,transaction,378,23.93,NaN,NaN,NaN,NaN


In [49]:
person_id = 'f367a50b86d049799bbb0eb645ee834c'

display(
    df_merge[df_merge['person'] == person_id]
    .sort_values('time')
)

,Unnamed: 0,person,event,time,offer_id,offer_type,reward,difficulty,duration
12560,12560,f367a50b86d049799bbb0eb645ee834c,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10.0,10.0,5.0
15540,15540,f367a50b86d049799bbb0eb645ee834c,transaction,0,195.24,NaN,NaN,NaN,NaN
15541,15541,f367a50b86d049799bbb0eb645ee834c,offer completed,0,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10.0,10.0,5.0
29487,29487,f367a50b86d049799bbb0eb645ee834c,offer viewed,42,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10.0,10.0,5.0
40765,40765,f367a50b86d049799bbb0eb645ee834c,transaction,90,1.63,NaN,NaN,NaN,NaN
65766,65766,f367a50b86d049799bbb0eb645ee834c,offer received,168,3f207df678b143eea3cee63160fa8bed,informational,0.0,0.0,4.0
100241,100241,f367a50b86d049799bbb0eb645ee834c,offer viewed,270,3f207df678b143eea3cee63160fa8bed,informational,0.0,0.0,4.0
123452,123452,f367a50b86d049799bbb0eb645ee834c,offer received,336,2906b810c7d4411798c6938adc9daaa5,discount,2.0,10.0,7.0
137409,137409,f367a50b86d049799bbb0eb645ee834c,offer viewed,360,2906b810c7d4411798c6938adc9daaa5,discount,2.0,10.0,7.0
163287,163287,f367a50b86d049799bbb0eb645ee834c,offer received,408,3f207df678b143eea3cee63160fa8bed,informational,0.0,0.0,4.0


In [50]:
first_event_time0 = (
    df_merge[df_merge['time'] == 0]
    .sort_values(['person', 'time'])
    .groupby('person')
    .first()
)

not_received_time0 = first_event_time0[first_event_time0['event'] != 'offer received']

print("time==0인데 received 아닌 사람 수:", len(not_received_time0))
display(not_received_time0[['event']].value_counts())

time==0인데 received 아닌 사람 수: 111


event      
transaction    111
Name: count, dtype: int64

In [51]:
!jupyter nbconvert --to markdown "02_eda_v1.ipynb"

[NbConvertApp] Converting notebook 02_eda_v1.ipynb to markdown
[NbConvertApp] Writing 38525 bytes to 02_eda_v1.md
